## Local LLM?
- ChatGPT 같은 외부 클라우드 API에 요청을 보내는 방식이 아니라, 사용자의 PC,서버,GPU 환경에서 직접 실행하는 대규모 언어 모델이다.
- 질문, 파일 등을 외부 서버로 보내지 않고 로컬 환경에서 모델을 불러와 추론한다.
- 로컬 LLM 실행 방식
    1. 개인 PC에서 Ollama로 qwen, llama, gemma 모델 실행
    2. GPU 서버에서 vLLM으로 LLM API 서버 실행
    3. CPU 또는 저사양 장비에서 llama.cpp 기반 GGUF 모델 실행
    4. Python 코드를 활용하여 Hugging Face Transformers로 모델 직접 로딩

## Local LLM과 클라우드 LLM API의 차이

| 구분      | Local LLM                          | 클라우드 LLM API                       |
| ------- | ---------------------------------- | ---------------------------------- |
| 실행 위치   | 내 PC 또는 내부 서버                      | 외부 서비스 제공사 서버                      |
| 대표 예시   | Ollama, vLLM, llama.cpp, LM Studio | OpenAI API, Claude API, Gemini API |
| 인터넷 의존도 | 모델 다운로드 이후 오프라인 사용 가능              | 대부분 인터넷 필요                         |
| 데이터 처리  | 입력 데이터가 내부 환경에 머무름                 | 외부 API 서버로 전송                      |
| 성능      | 장비 사양과 모델 크기에 따라 달라짐               | 제공사 인프라에 따라 안정적                    |
| 비용 구조   | 초기 장비 비용 중심                        | 사용량 기반 과금 중심                       |
| 관리 난이도  | 설치, 모델, GPU, 메모리 관리 필요             | API Key 중심으로 비교적 간단                |

## Local LLM을 사용하는 이유
### 1. 데이터 보안
- Local LLM은 입력 데이터가 외부로 전송되지 않기 때문에, 회사의 내부 문서, 기업 데이터, 연구자료처럼 외부 전송이 부담되는 데이터를 다루기 좋다.
### 2. 비용 문제
- 클라우드 LLM은 요청 수와 토큰 사용량에 따라 비용 발생, Local LLM은 장비와 운영 환경을 갖추고 나면 반복적이거나 대량의 테스트를 자유롭게 수행할 수 있음
### 3. 오프라인 또는 내부망 활용
### 4. 모델 제어 가능
- 원하는 모델을 직접 선택하고 교체할 수 있다.

### Local LLM의 단점
- 모델 크기에 따라 GPU VRAM, RAM, 저장공간이 많이 필요하기 때문에 구축에 많은 비용이 소비된다.
- 고성능 클라우드 API 모델보다 응답 품질이 떨어질 수 있음
- 동시 사용자 수가 많아질수록 응답 속도가 달라질 수 있기 때문에 추가적인 인력 비용이 들어갈 수 있다.

## Ollama
- 2023년경 공개적으로 알려지기 시작한 로컬 LLM 실행 도구
- Local LLM을 쉽게 실행할 수 있도록 도와주는 도구, 모델 다운로드, 실행, API 서버 제공을 간단한 명령어로 처리할 수 있다.

In [55]:
# 압축해제에 필요한 라이브러리 설치
!apt-get update
!apt-get install -y zstd

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to r

In [56]:
# Ollama 설치
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [57]:
# 설치 버전확인
!ollama --version

ollama version is 0.31.1


## Ollama 실행하기
- Ollama는 백그라운드 서버가 실행되어 있어야 LangChain 또는 API에서 모델을 호출할 수 있음

In [58]:
# Ollama 구동하기
!nohup ollama serve > ollama.log 2>%1 &

In [59]:
# 모델정보 확인하기
!curl http://127.0.0.1:11434/api/tags

{"models":[{"name":"qwen3:0.6b","model":"qwen3:0.6b","modified_at":"2026-07-01T06:32:48.365162232Z","size":522653767,"digest":"7df6b6e09427a769808717c0a93cadc4ae99ed4eb8bf5ca557c90846becea435","details":{"parent_model":"","format":"gguf","family":"qwen3","families":["qwen3"],"parameter_size":"751.63M","quantization_level":"Q4_K_M","context_length":40960,"embedding_length":1024},"capabilities":["completion","tools","thinking"]},{"name":"qwen3:4b","model":"qwen3:4b","modified_at":"2026-07-01T06:28:50.491188396Z","size":2497293931,"digest":"359d7dd4bcdab3d86b87d73ac27966f4dbb9f5efdfcc75d34a8764a09474fae7","details":{"parent_model":"","format":"gguf","family":"qwen3","families":["qwen3"],"parameter_size":"4.0B","quantization_level":"Q4_K_M","context_length":262144,"embedding_length":2560},"capabilities":["completion","tools","thinking"]}]}

In [60]:
!ollama list

NAME          ID              SIZE      MODIFIED       
qwen3:0.6b    7df6b6e09427    522 MB    9 minutes ago     
qwen3:4b      359d7dd4bcda    2.5 GB    13 minutes ago    


###모델 설치
- https://ollama.com//search

In [61]:
!ollama pull qwen3:4b

In [62]:
!ollama list

NAME          ID              SIZE      MODIFIED               
qwen3:4b      359d7dd4bcda    2.5 GB    Less than a second ago    
qwen3:0.6b    7df6b6e09427    522 MB    9 minutes ago             


In [63]:
!ollama pull qwen3:0.6b

In [64]:
!ollama list

NAME          ID              SIZE      MODIFIED               
qwen3:0.6b    7df6b6e09427    522 MB    Less than a second ago    
qwen3:4b      359d7dd4bcda    2.5 GB    1 second ago              


### 직접 호출하기

In [65]:
import requests

In [66]:
info = {
    'model' : "qwen3:0.6b", #사용할 모델
    'prompt' : "너는 여행 전문가야. 한국의 여행지 중에 아름다운 곳 3가지 추천해줘. 모든 응답은 한글로 작성해줘",
    'stream' : False # invoke처럼 응답을 한번에 JSON으로 받음
}

In [67]:
try:
  response = requests.post('http://localhost:11434/api/generate', json=info, timeout=120 )
  # 응답코드가 정상(200)이 아니면 예외를 발생
  response.raise_for_status()
  # 응답 JSON 가져오기
  data = response.json()
  print(data.get("response"))
except Exception as e:
  print(e)

1. **서울 (Seoul)**: 아름다운 곳으로, 현대적인 도시와 전통적인 문화를 보 드릴 수 있는 곳입니다. 산업화된 아일랜드와 협동된 관광지로, 다양한 관광 삶의 경험을 즐기실 수 있습니다.  
2. **고려남 (Gyeongnam)**: 전통과 현대의 조화를 자랑하는 곳으로, 역사적인 장소와 신화를 상징하는 아름다운 자연환경을 만나실 수 있습니다.  
3. **대전 (Daejeon)**: 전통 마을과 현대 도시의 상징적인 장소를 모두 보 드릴 수 있는 곳입니다. 전통 마을의 전통시장과 현대적인 관광지로, 다양한 경험을 즐기실 수 있습니다.


In [68]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [ ]:
# 올라마 프로세스 끄기
!pkill -f "ollama serve"

## Ollama_LangChain 구성하기